# ADAS Lane2OpenDRIVE Colab

Run the cells from top to bottom on a Google Colab **T4 GPU**. This notebook uses the official Ultra-Fast-Lane-Detection CULane pretrained 2D lane detector for arbitrary videos. It never creates dummy detections or fabricated metric values.

Metric distance, calibrated visual-motion speed, and OpenDRIVE require real camera calibration values in `configs/camera.yaml`. Without them, the run safely produces image-space lanes and reports `NON_METRIC`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/vamos-sujal/lane-opendrive.git'
REPO_DIR = Path('/content/lane-opendrive')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', Path.cwd())

In [ ]:
import subprocess
import sys
import torch

print(sys.version)
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.is_available(), 'CUDA is unavailable. In Colab choose Runtime > Change runtime type > T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name, 'CUDA:', torch.version.cuda, 'PyTorch:', torch.__version__)
assert 'T4' in gpu_name, f'Expected a T4 GPU, got {gpu_name}. Select a T4 runtime and rerun from the first cell.'

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.chdir('/content/lane-opendrive')
def run(command):
    print('$', ' '.join(command))
    subprocess.run(command, check=True)

print('Python:', sys.version)
if sys.version_info < (3, 10):
    raise RuntimeError('Python 3.10 or newer is required.')

# Install wheels only. This prevents long source builds on Colab Python 3.13.
colab_packages = [
    'numpy', 'scipy', 'opencv-python', 'matplotlib', 'PyYAML', 'lxml',
    'shapely', 'pytest', 'pandas', 'xmlschema', 'networkx', 'gdown',
    'Pillow', 'addict==2.4.0', 'pathspec==0.12.1',
]
run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--only-binary=:all:'] + colab_packages)

# Preserve Colab's CUDA-enabled PyTorch installation.
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'CUDA is required for the T4 lane detector.'

if Path('/content/Ultra-Fast-Lane-Detection').exists():
    import shutil
    shutil.rmtree('/content/Ultra-Fast-Lane-Detection')
run(['git', 'clone', '--depth', '1', 'https://github.com/cfzd/Ultra-Fast-Lane-Detection.git', '/content/Ultra-Fast-Lane-Detection'])
run([sys.executable, '/content/lane-opendrive/scripts/verify_environment.py'])
print('UFLD source ready: /content/Ultra-Fast-Lane-Detection')
print('Environment verification passed.')

In [ ]:
import json
import os
import subprocess
import sys
import yaml
from pathlib import Path

REPO_DIR = Path('/content/lane-opendrive')
os.chdir(REPO_DIR)
MODEL_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_config = yaml.safe_load(Path('/content/lane-opendrive/configs/model.yaml').read_text())
checkpoint_config = model_config['checkpoint']
checkpoint_name = Path(checkpoint_config['path']).name
CHECKPOINT_PATH = MODEL_DIR / checkpoint_name
MANIFEST_PATH = MODEL_DIR / 'manifest.json'
expected_url = checkpoint_config['url']

# Never reuse a checkpoint downloaded from a different model URL.
if CHECKPOINT_PATH.exists():
    manifest_url = None
    if MANIFEST_PATH.exists():
        try:
            manifest_url = json.loads(MANIFEST_PATH.read_text()).get('url')
        except json.JSONDecodeError:
            pass
    if manifest_url != expected_url:
        print('Removing stale or unverified checkpoint:', CHECKPOINT_PATH)
        CHECKPOINT_PATH.unlink(missing_ok=True)
        MANIFEST_PATH.unlink(missing_ok=True)

if not CHECKPOINT_PATH.exists():
    subprocess.run([
        sys.executable, '/content/lane-opendrive/scripts/download_weights.py',
        '--config', '/content/lane-opendrive/configs/model.yaml',
        '--output-dir', str(MODEL_DIR),
    ], check=True, cwd=str(REPO_DIR))
else:
    print('Reusing verified CULane checkpoint from Drive:', CHECKPOINT_PATH)

assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'
smoke = subprocess.run([
    sys.executable, '/content/lane-opendrive/scripts/smoke_test.py',
    '--device', 'cuda',
    '--checkpoint', str(CHECKPOINT_PATH),
    '--repo-path', '/content/Ultra-Fast-Lane-Detection',
], cwd=str(REPO_DIR), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(smoke.stdout)
if smoke.returncode != 0:
    raise RuntimeError(f'UFLD CULane checkpoint smoke test failed with exit code {smoke.returncode}.')
print('Loaded detector: official UFLD CULane checkpoint loaded and warmed up')

In [ ]:
from pathlib import Path
from google.colab import files

INPUT_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/input')
RUNS_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/runs')
MODEL_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/models')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi'}
def find_videos():
    return sorted(path for path in INPUT_DIR.iterdir() if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS)

video_files = find_videos()
if not video_files:
    print('No video found in Drive. Upload one video now.')
    uploaded = files.upload()
    for name, content in uploaded.items():
        destination = INPUT_DIR / Path(name).name
        destination.write_bytes(content)
    video_files = find_videos()

if not video_files:
    raise FileNotFoundError('No .mp4, .mov, or .avi video is available in the Drive input folder.')
if len(video_files) > 1 and not (INPUT_DIR / 'input.mp4').exists():
    raise RuntimeError('Multiple videos found. Keep one video or rename the intended file to input.mp4.')

VIDEO_PATH = str(INPUT_DIR)
print('Video selected:', INPUT_DIR / 'input.mp4' if (INPUT_DIR / 'input.mp4').exists() else video_files[0])

In [ ]:
import datetime
import os
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/lane-opendrive')
os.chdir(REPO_DIR)
run_id = datetime.datetime.now(datetime.timezone.utc).strftime('run_%Y%m%dT%H%M%SZ')
RUN_DIR = RUNS_DIR / run_id
subprocess.run([
    'python', '/content/lane-opendrive/scripts/run_video.py',
    '--input', VIDEO_PATH,
    '--output', str(RUN_DIR),
    '--config', '/content/lane-opendrive/configs/camera.yaml',
    '--detector', 'ufld_culane',
    '--checkpoint', str(CHECKPOINT_PATH),
], check=True, cwd=str(REPO_DIR))

In [ ]:
import json
from IPython.display import Image, Video, display

summary_path = RUN_DIR / 'run_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('=== VIDEO REPORT ===')
    print(json.dumps(summary, indent=2))
    print('\n=== INTERPRETATION ===')
    print('Lane count and IDs are detector/tracker outputs.')
    print('Metric distance and speed appear only with trusted camera calibration.')
    print('OpenDRIVE is emitted only after metric geometry and XML validation pass.')

report_names = (
    'input_metadata.json', 'metric_report.json', 'tracking_results.json',
    'lane_graph.json', 'geometry.json', 'validation_report.json',
)
for name in report_names:
    path = RUN_DIR / name
    if path.exists():
        print(f'\n{name}')
        print(json.dumps(json.loads(path.read_text()), indent=2))

visual_dir = RUN_DIR / 'visualizations'
for name in ('original_video.mp4', 'lane_overlay.mp4', 'topology_overlay.mp4', 'top_down.mp4'):
    path = visual_dir / name
    if not path.is_file() or path.stat().st_size == 0:
        print(f'Skipping unavailable video: {name}')
        continue
    print(f'\n{name} ({path.stat().st_size / 1024 / 1024:.1f} MB)')
    try:
        display(Video(filename=str(path), embed=True, width=960))
    except Exception as exc:
        print(f'Preview unavailable for {name}: {type(exc).__name__}: {exc}')
        print(f'File remains available at: {path}')

for path in sorted(visual_dir.glob('*_frame_*.jpg')):
    display(Image(filename=str(path)))

xodr = RUN_DIR / 'output.xodr'
print('\nOpenDRIVE:', xodr if xodr.exists() else 'not produced: metric validation failed closed')

In [ ]:
from IPython.display import display, Image
for image in sorted((RUN_DIR / 'visualizations').glob('*.jpg')):
    print(image.name)
    display(Image(filename=str(image)))